# Label image for YOLOv8s

In [1]:
import cv2
import numpy as np
import os
import shutil

# Root folders
input_root = r"D:/Pill_Identification/dataset/For_model/Pill_jpg_2025_Mask"
output_root = r"D:/Pill_Identification/model/YOLOv8/Pill_YOLO_Labels"
output_boundary_root = r"D:/Pill_Identification/model/YOLOv8/Pill_YOLO_bounding_boxes"

# Remove output folders if they exist
if os.path.exists(output_root):
    shutil.rmtree(output_root)
if os.path.exists(output_boundary_root):
    shutil.rmtree(output_boundary_root)

# Create fresh output folders
os.makedirs(output_root, exist_ok=True)
os.makedirs(output_boundary_root, exist_ok=True)

# Supported image extensions
image_extensions = ('.png', '.jpg', '.jpeg')

for root, dirs, files in os.walk(input_root):
    for file in files:
        if file.lower().endswith(image_extensions):
            input_path = os.path.join(root, file)

            # Filename without extension
            filename_wo_ext = os.path.splitext(file)[0]

            # Output paths (NO subfolders, all in a flat directory)
            txt_output_path = os.path.join(output_root, filename_wo_ext + ".txt")
            bbox_output_path = os.path.join(output_boundary_root, filename_wo_ext + "_bbox.png")

            # Load mask
            mask = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                print(f"❌ Failed to load image: {input_path}")
                continue

            # Threshold to binary
            _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

            # Find contours
            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            height, width = binary.shape
            output_img = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)

            label_lines = []

            # Extract class ID from filename prefix
            class_id = filename_wo_ext.split('_')[0]

            for cnt in contours:
                x, y, w, h = cv2.boundingRect(cnt)

                # Convert to YOLO format
                x_center = (x + w / 2) / width
                y_center = (y + h / 2) / height
                w_norm = w / width
                h_norm = h / height

                yolo_format = f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
                label_lines.append(yolo_format)

                # Draw bounding box
                cv2.rectangle(output_img, (x, y), (x + w, y + h), (0, 255, 0), 2)

            # Save YOLO label file
            with open(txt_output_path, "w") as f:
                for line in label_lines:
                    f.write(line)  # ensure newline per label

            # Save image with bounding boxes
            cv2.imwrite(bbox_output_path, output_img)
            print(f"✅ Processed: {file}")

✅ Processed: 0_MS_01.png
✅ Processed: 0_MS_02.png
✅ Processed: 0_MS_03.png
✅ Processed: 0_MS_04.png
✅ Processed: 0_MS_05.png
✅ Processed: 0_MS_06.png
✅ Processed: 0_MS_07.png
✅ Processed: 0_MS_08.png
✅ Processed: 0_MS_09.png
✅ Processed: 0_MT_01.png
✅ Processed: 0_MT_02.png
✅ Processed: 0_MT_03.png
✅ Processed: 0_MT_04.png
✅ Processed: 0_MT_05.png
✅ Processed: 0_MT_06.png
✅ Processed: 0_MT_07.png
✅ Processed: 0_MT_08.png
✅ Processed: 0_MT_09.png
✅ Processed: 100_MS_01.png
✅ Processed: 100_MS_02.png
✅ Processed: 100_MS_03.png
✅ Processed: 100_MS_04.png
✅ Processed: 100_MS_05.png
✅ Processed: 100_MS_06.png
✅ Processed: 100_MS_07.png
✅ Processed: 100_MS_08.png
✅ Processed: 100_MS_09.png
✅ Processed: 100_MT_01.png
✅ Processed: 100_MT_02.png
✅ Processed: 100_MT_03.png
✅ Processed: 100_MT_04.png
✅ Processed: 100_MT_05.png
✅ Processed: 100_MT_06.png
✅ Processed: 100_MT_07.png
✅ Processed: 100_MT_08.png
✅ Processed: 100_MT_09.png
✅ Processed: 101_MS_01.png
✅ Processed: 101_MS_02.png
✅ Processe

In [2]:
with open("layer_indices.txt", "w") as f:
    for i in range(378):
        f.write(f"  {i}: {i}\n")

# Label image for RetinaNet

In [1]:
import cv2
import numpy as np
import os
import shutil
import csv

# Root folders
input_root = "D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results"
csv_output_path = "D:/Pill_Identification/RetinaNet/pill_bounding_boxes.csv"

# Remove existing CSV
if os.path.exists(csv_output_path):
    os.remove(csv_output_path)

# Supported image extensions
image_extensions = ('.png', '.jpg', '.jpeg')

# Open CSV writer
with open(csv_output_path, mode='w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['image_id', 'width', 'height', 'class_id', 'x', 'y', 'w', 'h'])

    for root, dirs, files in os.walk(input_root):
        for file in files:
            if file.lower().endswith(image_extensions):
                input_path = os.path.join(root, file)
                filename_wo_ext = os.path.splitext(file)[0]
                image_id = filename_wo_ext

                # Load mask
                mask = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
                if mask is None:
                    print(f"❌ Failed to load image: {input_path}")
                    continue

                # Threshold to binary
                _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

                # Find contours
                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                height, width = binary.shape

                # Extract class ID from filename (e.g., 0_MS_01.png → class_id = 0)
                class_id = filename_wo_ext.split('_')[0]

                for cnt in contours:
                    x, y, w, h = cv2.boundingRect(cnt)

                    # Convert to float
                    x_f = float(x)
                    y_f = float(y)
                    w_f = float(w)
                    h_f = float(h)

                    # Write row
                    csv_writer.writerow([image_id, width, height, class_id, x_f, y_f, w_f, h_f])

                print(f"✅ Processed: {input_path}")


✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results\0_MS_01.png
✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results\0_MS_02.png
✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results\0_MT_01.png
✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results\0_MT_02.png
✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results\0_MT_03.png
✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results\10_MS_101.png
✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results\10_MS_102.png
✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results\10_MS_103.png
✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_results\10_MT_101.png
✅ Processed: D:/Pill_Identification/background_removal_DL/test_data/images/u2net_re